# Genetischer Algorithmus: Das Rucksackproblem lösen

In diesem Notebook implementierst du einen genetischen Algorithmus in Python, der das Rucksackproblem aus der Theorie-Seite löst: 4 Gegenstände (A, B, C, D) mit unterschiedlichem Wert und Gewicht sollen so in einen Rucksack gepackt werden, dass der Gesamtwert möglichst hoch ist, ohne eine Gewichtsgrenze von 50 zu überschreiten.

Du setzt dabei Schritt für Schritt die Bausteine aus dem Ablaufschema um: Population, Fitnessfunktion, Selektion, Crossover und Mutation.

Zuerst installierst du die benötigte Bibliothek für die Diagramme:

In [ ]:
%pip install matplotlib

Danach importierst du die benötigten Module. `random` ist Teil der Python-Standardbibliothek und wird für alle Zufallsentscheidungen (Initialisierung, Selektion, Crossover, Mutation) gebraucht.

In [ ]:
import random
import matplotlib.pyplot as plt

# Zufallsgenerator festlegen, damit die Ergebnisse reproduzierbar sind
random.seed(42)

### Schritt 1: Das Problem definieren

Genau wie auf der Theorie-Seite: vier Gegenstände mit Wert und Gewicht, plus die Gewichtsgrenze des Rucksacks.

In [ ]:
# Gegenstände in der Reihenfolge A, B, C, D (wie auf der Theorie-Seite)
werte = [60, 100, 120, 80]
gewichte = [10, 20, 30, 15]
gewichtsgrenze = 50

anzahl_gegenstaende = len(werte)
print(f"Anzahl Gegenstände: {anzahl_gegenstaende}")

### Schritt 2: Die Fitnessfunktion

Ein Chromosom ist ein Bitstring der Länge 4, z. B. `[1, 1, 0, 0]` bedeutet "packe A und B ein". Die Fitness ist die Wertsumme der eingepackten Gegenstände – außer das Gesamtgewicht überschreitet die Grenze, dann ist die Fitness 0 (genau wie im Beispiel auf der Theorie-Seite).

In [ ]:
def fitness(chromosom):
    gesamtwert = 0
    gesamtgewicht = 0
    for i in range(anzahl_gegenstaende):
        if chromosom[i] == 1:
            gesamtwert += werte[i]
            gesamtgewicht += gewichte[i]

    if gesamtgewicht > gewichtsgrenze:
        return 0  # Strafe: zu schwer, daher Fitness 0
    return gesamtwert


# Test mit den beiden Beispiel-Chromosomen von der Theorie-Seite
print("Chromosom [1,1,0,0]:", fitness([1, 1, 0, 0]))  # sollte 160 ergeben
print("Chromosom [1,0,1,1]:", fitness([1, 0, 1, 1]))  # sollte 0 ergeben (zu schwer)

### Aufgabe 1:
Die Funktion `fitness` ist oben bereits vollständig fertig. Nutze sie, um die Fitness der beiden Chromosomen aus Aufgabe 1 der Theorie-Seite zu berechnen: Chromosom 3 = `[0, 1, 1, 0]` und Chromosom 4 = `[1, 1, 0, 1]`. Vergleiche das Ergebnis mit deiner Rechnung von Hand.

In [ ]:
# Hier deinen Code einfügen:
# print("Chromosom 3:", fitness([0, 1, 1, 0]))
# print("Chromosom 4:", fitness([1, 1, 0, 1]))


### Schritt 3: Population initialisieren

Eine Population ist eine Liste von zufällig erzeugten Chromosomen (Bitstrings der Länge 4).

In [ ]:
def erzeuge_population(groesse):
    population = []
    for _ in range(groesse):
        chromosom = [random.randint(0, 1) for _ in range(anzahl_gegenstaende)]
        population.append(chromosom)
    return population


# Beispiel: eine kleine Population von 6 Chromosomen erzeugen und anzeigen
beispiel_population = erzeuge_population(6)
for c in beispiel_population:
    print(c, "-> Fitness:", fitness(c))

### Schritt 4: Selektion

Wir verwenden hier eine einfache **Turnierselektion**: Es werden zufällig 3 Chromosomen aus der Population gezogen, und das mit der besten Fitness gewinnt das "Turnier" und darf sich fortpflanzen. Das sorgt dafür, dass bessere Chromosomen häufiger ausgewählt werden, ohne dass ausschließlich das allerbeste immer gewinnt.

In [ ]:
def selektiere(population, turniergroesse=3):
    teilnehmer = random.sample(population, turniergroesse)
    teilnehmer.sort(key=fitness, reverse=True)
    return teilnehmer[0]  # Chromosom mit der besten Fitness gewinnt

### Schritt 5: Crossover

Beim Ein-Punkt-Crossover wird eine zufällige Position im Chromosom gewählt. Das erste Kind erhält den Anfang von Elternteil 1 und das Ende von Elternteil 2, das zweite Kind umgekehrt.

In [ ]:
def crossover(eltern1, eltern2):
    position = random.randint(1, anzahl_gegenstaende - 1)
    kind1 = eltern1[:position] + eltern2[position:]
    kind2 = eltern2[:position] + eltern1[position:]
    return kind1, kind2


# Beispiel
beispiel_kind1, beispiel_kind2 = crossover([1, 1, 0, 0], [0, 0, 1, 1])
print("Kind 1:", beispiel_kind1)
print("Kind 2:", beispiel_kind2)

### Schritt 6: Mutation

Mit einer festgelegten Wahrscheinlichkeit (Mutationsrate) wird jedes einzelne Bit eines Chromosoms umgedreht (aus 0 wird 1, aus 1 wird 0).

In [ ]:
def mutiere(chromosom, mutationsrate):
    neues_chromosom = chromosom.copy()
    for i in range(len(neues_chromosom)):
        if random.random() < mutationsrate:
            neues_chromosom[i] = 1 - neues_chromosom[i]  # 0 wird zu 1, 1 wird zu 0
    return neues_chromosom

### Schritt 7: Der genetische Algorithmus

Jetzt setzen wir alle Bausteine zum vollständigen Ablauf zusammen (vergleiche das Ablaufschema in Abschnitt 2 der Theorie-Seite): Population initialisieren, dann über mehrere Generationen hinweg selektieren, kombinieren, mutieren und die neue Generation bilden.

Zwei Verbesserungen sorgen dafür, dass gute Lösungen nicht wieder verloren gehen:
- **Elitismus:** Die `elite` besten Chromosomen einer Generation werden unverändert (ohne Crossover und Mutation) in die nächste Generation übernommen. Dadurch kann die beste Fitness der Population nicht mehr sinken.
- **Bestes Chromosom merken:** Wir speichern das beste jemals gefundene Chromosom. Am Ende wird dieses zurückgegeben – nicht nur das beste der letzten Generation.

In [ ]:
def genetischer_algorithmus(populationsgroesse, anzahl_generationen, mutationsrate, elite=2):
    population = erzeuge_population(populationsgroesse)
    beste_fitness_pro_generation = []
    bestes_jemals = max(population, key=fitness)

    for generation in range(anzahl_generationen):
        # Population nach Fitness sortieren (beste zuerst)
        population.sort(key=fitness, reverse=True)

        # Beste Fitness dieser Generation merken
        beste_fitness_pro_generation.append(fitness(population[0]))

        # Bestes jemals gefundenes Chromosom aktualisieren
        if fitness(population[0]) > fitness(bestes_jemals):
            bestes_jemals = population[0].copy()

        # Elitismus: die besten Chromosomen unverändert übernehmen
        neue_population = [c.copy() for c in population[:elite]]

        # Rest der neuen Generation durch Selektion, Crossover und Mutation erzeugen
        while len(neue_population) < populationsgroesse:
            eltern1 = selektiere(population)
            eltern2 = selektiere(population)
            kind1, kind2 = crossover(eltern1, eltern2)
            neue_population.append(mutiere(kind1, mutationsrate))
            neue_population.append(mutiere(kind2, mutationsrate))

        population = neue_population[:populationsgroesse]

    # Auch die letzte Generation noch prüfen
    kandidat = max(population, key=fitness)
    if fitness(kandidat) > fitness(bestes_jemals):
        bestes_jemals = kandidat.copy()

    return bestes_jemals, beste_fitness_pro_generation

In [ ]:
# Genetischen Algorithmus mit Beispiel-Parametern ausführen
bestes_chromosom, fitness_verlauf = genetischer_algorithmus(
    populationsgroesse=20,
    anzahl_generationen=30,
    mutationsrate=0.1
)

print("Bestes gefundenes Chromosom:", bestes_chromosom)
print("Fitness (Gesamtwert):", fitness(bestes_chromosom))

eingepackt = [chr(65 + i) for i in range(anzahl_gegenstaende) if bestes_chromosom[i] == 1]
print("Eingepackte Gegenstände:", eingepackt)

### Schritt 8: Fitness-Verlauf plotten

Das folgende Diagramm zeigt, wie sich die beste Fitness der Population über die Generationen hinweg entwickelt. Dank Elitismus kann die Kurve nicht mehr fallen.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(fitness_verlauf, marker='o')
plt.title('Beste Fitness pro Generation')
plt.xlabel('Generation')
plt.ylabel('Beste Fitness (Gesamtwert)')
plt.grid(True)
plt.show()

### Aufgabe 2:
Der Code unten ist bereits vollständig lauffähig und entspricht genau dem Aufruf oben. Verändere **nur die markierte Zeile** (die Mutationsrate) und führe die Zelle mehrfach mit unterschiedlichen Werten aus:

1. Zuerst mit einer sehr **niedrigen** Mutationsrate, z. B. `0.01`
2. Dann mit einer sehr **hohen** Mutationsrate, z. B. `0.5`

Vergleiche jeweils den Diagrammverlauf: Bei welcher Mutationsrate konvergiert die beste Fitness schnell und stabil, bei welcher bleibt sie eher sprunghaft oder verbessert sich kaum?

In [ ]:
mutationsrate_test = 0.1  # <-- HIER ÄNDERST DU: z. B. auf 0.01 oder 0.5 setzen

bestes_chromosom_test, fitness_verlauf_test = genetischer_algorithmus(
    populationsgroesse=20,
    anzahl_generationen=30,
    mutationsrate=mutationsrate_test
)

print(f"Mutationsrate = {mutationsrate_test}")
print("Bestes gefundenes Chromosom:", bestes_chromosom_test, "- Fitness:", fitness(bestes_chromosom_test))

plt.figure(figsize=(8, 5))
plt.plot(fitness_verlauf_test, marker='o')
plt.title(f'Beste Fitness pro Generation (Mutationsrate = {mutationsrate_test})')
plt.xlabel('Generation')
plt.ylabel('Beste Fitness (Gesamtwert)')
plt.grid(True)
plt.show()

### Aufgabe 3:
Experimentiere jetzt mit der **Populationsgröße** und der **Anzahl der Generationen** statt der Mutationsrate. Kopiere dazu die Code-Zelle von Aufgabe 2 in die Zelle unten und ändere `populationsgroesse` bzw. `anzahl_generationen`. Findet der Algorithmus mit einer kleineren Population (z. B. 6) genauso zuverlässig die beste Lösung wie mit einer größeren (z. B. 50)?

**Zusatz:** Setze beim Aufruf zusätzlich `elite=0` (kein Elitismus) und vergleiche den Fitness-Verlauf mit `elite=2`.

In [ ]:
# Hier deinen Code einfügen (kopiere die Struktur aus Aufgabe 2 und ändere populationsgroesse / anzahl_generationen)


## Zusammenfassung

In diesem Notebook hast du:

1. das Rucksackproblem als Chromosom-Codierung und Fitnessfunktion definiert
2. eine Population zufällig initialisiert
3. Selektion (Turnierselektion), Crossover (Ein-Punkt) und Mutation implementiert
4. alle Bausteine zu einem vollständigen genetischen Algorithmus mit Elitismus zusammengesetzt
5. den Fitness-Verlauf über die Generationen visualisiert
6. den Einfluss von Mutationsrate, Populationsgröße und Generationenanzahl auf die Konvergenz untersucht

Damit hast du den kompletten Ablauf aus der Theorie-Seite (Population → Fitness → Selektion → Crossover → Mutation → neue Generation) praktisch nachvollzogen.